# 03. Off-Policy vs. On-Policy Distillation Algorithms

This notebook illustrates the distillation algorithms implemented in `distillfw`:
- **Forward KL (`forward_kl`)**: Classic word-level mode-covering KD (Hinton et al., 2015).
- **Reverse KL (`reverse_kl`)**: Mode-seeking KD avoiding student hallucinations in low-density teacher regions (MiniLLM).
- **Skew KL (`skew_kl`)**: Interpolated divergence $\text{KL}(p \parallel \alpha p + (1-\alpha)q_\theta)$ for bounded gradients (DistiLLM, ICML 2024).
- **Contrastive Asymmetric KD (`distillm2`)**: Skew KLD on teacher trajectories + Reverse KLD on offline student trajectories (DistiLLM-2, ICML 2025).
- **Sparse Top-K Logprob KD (`sparse_topk_kl_loss`)**: Gray-box API distillation using Gemini `response_logprobs`.
- **Generalized KD (`gkd`)**: Hybrid/On-policy student rollout mixing via `gkd_lambda`.

# 03. Off-Policy vs. On-Policy Distillation Algorithms

This notebook illustrates the distillation algorithms implemented in `distillfw`:
- **Forward KL (`forward_kl`)**: Classic word-level mode-covering KD (Hinton et al., 2015).
- **Reverse KL (`reverse_kl`)**: Mode-seeking KD avoiding student hallucinations in low-density teacher regions (MiniLLM).
- **Skew KL (`skew_kl`)**: Interpolated divergence $\text{KL}(p \parallel \alpha p + (1-\alpha)q_\theta)$ for bounded gradients (DistiLLM, ICML 2024).
- **Contrastive Asymmetric KD (`distillm2`)**: Skew KLD on teacher trajectories + Reverse KLD on offline student trajectories (DistiLLM-2, ICML 2025).
- **Sparse Top-K Logprob KD (`sparse_topk_kl_loss`)**: Gray-box API distillation using Gemini `response_logprobs`.
- **Generalized KD (`gkd`)**: Hybrid/On-policy student rollout mixing via `gkd_lambda`.

In [ ]:
import torch
from distillfw.algorithms.losses import (
    forward_kl_loss,
    reverse_kl_loss,
    jensen_shannon_loss,
    skew_kl_loss,
    distillm2_contrastive_loss,
    sparse_topk_kl_loss,
)

batch_size, seq_len, vocab_size = 2, 8, 128
student_logits = torch.randn(batch_size, seq_len, vocab_size, requires_grad=True)
teacher_logits = torch.randn(batch_size, seq_len, vocab_size)
mask = torch.ones(batch_size, seq_len)

print("Forward KL Loss :", forward_kl_loss(student_logits, teacher_logits, mask).item())
print("Reverse KL Loss :", reverse_kl_loss(student_logits, teacher_logits, mask).item())
print("Jensen-Shannon  :", jensen_shannon_loss(student_logits, teacher_logits, mask, beta=0.5).item())
print("Skew KL (0.1)   :", skew_kl_loss(student_logits, teacher_logits, mask, alpha=0.1).item())
print(
    "DistiLLM-2 Loss :",
    distillm2_contrastive_loss(
        student_logits, teacher_logits, student_logits, teacher_logits, mask, mask
    ).item(),
)

# Gray-box API distillation from top-5 Gemini logprobs
topk_probs, topk_ids = torch.softmax(teacher_logits, dim=-1).topk(5, dim=-1)
print(
    "Sparse Top-K KL :",
    sparse_topk_kl_loss(student_logits, topk_ids, topk_probs.log(), mask, divergence="skew_kl").item(),
)

In [ ]:
import torch
from distillfw.algorithms.losses import (
    forward_kl_loss,
    reverse_kl_loss,
    jensen_shannon_loss,
    skew_kl_loss,
    distillm2_contrastive_loss,
    sparse_topk_kl_loss,
)

batch_size, seq_len, vocab_size = 2, 8, 128
student_logits = torch.randn(batch_size, seq_len, vocab_size, requires_grad=True)
teacher_logits = torch.randn(batch_size, seq_len, vocab_size)
mask = torch.ones(batch_size, seq_len)

print("Forward KL Loss :", forward_kl_loss(student_logits, teacher_logits, mask).item())
print("Reverse KL Loss :", reverse_kl_loss(student_logits, teacher_logits, mask).item())
print("Jensen-Shannon  :", jensen_shannon_loss(student_logits, teacher_logits, mask, beta=0.5).item())
print("Skew KL (0.1)   :", skew_kl_loss(student_logits, teacher_logits, mask, alpha=0.1).item())
print(
    "DistiLLM-2 Loss :",
    distillm2_contrastive_loss(
        student_logits, teacher_logits, student_logits, teacher_logits, mask, mask
    ).item(),
)

# Gray-box API distillation from top-5 Gemini logprobs
topk_probs, topk_ids = torch.softmax(teacher_logits, dim=-1).topk(5, dim=-1)
print(
    "Sparse Top-K KL :",
    sparse_topk_kl_loss(student_logits, topk_ids, topk_probs.log(), mask, divergence="skew_kl").item(),
)